# Классические NLP-задачи: Bag of Words, TF-IDF, n-граммы, Word2Vec, fastText

В этом ноутбуке собраны **5 классических задач по NLP**.

Для каждого метода мы:
1. **Реализуем алгоритм вручную** — на чистом numpy, чтобы понять формулы;
2. **Сравниваем** ручную реализацию с библиотечной (sklearn / gensim) и убеждаемся, что результат совпадает;
3. **Обучаем классификатор** и смотрим на метрики;
4. Разбираем **частые ошибки** и объясняем, в чём именно проблема.

Мы разберём:

| Метод | Задача | Ручная реализация |
|-------|--------|-------------------|
| **TF-IDF** | Тональность отзывов | TF, IDF, L2-нормализация |
| **Bag of Words** | Спам / не спам | Словарь + матрица счётчиков |
| **TF-IDF + n-граммы** | Intent classification | Извлечение биграмм + TF-IDF |
| **Word2Vec** | Классификация новостей | Skip-gram пары + обучение эмбеддингов |
| **fastText** | Определение языка | (библиотека — subword-подход) |


In [1]:
import math
import os
import random
import re
from collections import Counter
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity


## Общая идея

Мы специально используем **маленькие словари и простые тексты**.
Это позволяет увидеть саму механику методов — от самого простого к сложному:

| Метод | Задача | Ручная реализация |
|-------|--------|-------------------|
| **Bag of Words** | Спам / не спам | Словарь + матрица счётчиков |
| **TF-IDF** | Тональность отзывов | TF, IDF, L2-нормализация |
| **TF-IDF + n-граммы** | Intent classification | Извлечение биграмм + TF-IDF |
| **Word2Vec** | Классификация новостей | Skip-gram пары + обучение эмбеддингов |
| **fastText** | Определение языка | (библиотека — subword-подход) |

Каждый следующий метод добавляет что-то, чего не хватало предыдущему.
BoW не видит разницы между редким и частым словом — TF-IDF это чинит.
TF-IDF не видит «отменить заказ» как единое целое — n-граммы это чинят.
n-граммы не знают, что «матч» и «чемпионат» близки по смыслу — Word2Vec это чинит.

**Принцип ноутбука**: сначала реализуем метод вручную и проверяем,
что результат совпадает с библиотекой, затем строим полный пайплайн.


## Задача 1: Спам / не спам

Есть короткие SMS-сообщения.
Нужно определить:

- `spam`
- `ham`

Почему здесь естественен **Bag of Words**:

- сообщения короткие;
- часто достаточно просто знать, **какие слова встретились**;
- слова вроде `free`, `win`, `bonus`, `cash` хорошо маркируют спам;
- это одна из самых классических задач для CountVectorizer + MultinomialNB.


In [2]:
if not os.path.exists("sms_spam_small.csv"):
    rnd = random.Random(43)

    spam_starts = ["win", "free", "bonus", "urgent", "claim"]
    spam_mids = ["cash prize", "free ticket", "gift card", "bonus code", "promo"]
    spam_ends = ["call now", "click now", "limited offer", "reply now"]

    ham_starts = ["hi", "mom", "friend", "hello", "okay"]
    ham_mids = ["see you", "at home", "after work", "let us meet", "call me later"]
    ham_ends = ["tonight", "tomorrow", "soon", "when free"]

    rows = []
    for _ in range(220):
        text = f"{rnd.choice(spam_starts)} {rnd.choice(spam_mids)} {rnd.choice(spam_ends)}"
        rows.append((text, "spam"))
    for _ in range(220):
        text = f"{rnd.choice(ham_starts)} {rnd.choice(ham_mids)} {rnd.choice(ham_ends)}"
        rows.append((text, "ham"))

    rnd.shuffle(rows)
    pd.DataFrame(rows, columns=["text", "label"]).to_csv("sms_spam_small.csv", index=False)


In [4]:
df_sms = pd.read_csv("sms_spam_small.csv")
df_sms.head()


,text,label
0,hello after work tomorrow,ham
1,hello at home tomorrow,ham
2,claim cash prize call now,spam
3,urgent bonus code reply now,spam
4,okay call me later when free,ham


## Шаг 1: Реализуем Bag of Words вручную

**Bag of Words** — самое простое представление текста:

1. Собираем **словарь** — множество всех уникальных слов в корпусе.
2. Каждый документ превращается в вектор длины `|словарь|`.
3. В позиции $j$ стоит **количество вхождений** слова $j$ в этот документ.

Формально:

$$\text{BoW}(d, w) = \text{count}(w, d)$$

Порядок слов полностью теряется — отсюда и название «мешок слов».

In [5]:
class ManualBoW:
    """Bag of Words с нуля — повторяет логику sklearn CountVectorizer."""

    def fit_transform(self, texts):
        tokenized = [text.lower().split() for text in texts]
        self.vocab_ = sorted({w for tokens in tokenized for w in tokens})
        self.word_to_idx_ = {w: i for i, w in enumerate(self.vocab_)}
        return self._count(tokenized)

    def transform(self, texts):
        tokenized = [text.lower().split() for text in texts]
        return self._count(tokenized)

    def _count(self, tokenized):
        matrix = np.zeros((len(tokenized), len(self.vocab_)), dtype=int)
        for i, tokens in enumerate(tokenized):
            for w in tokens:
                if w in self.word_to_idx_:
                    matrix[i, self.word_to_idx_[w]] += 1
        return matrix


# --- Демонстрация ---
sms_sample = df_sms["text"].head(6).tolist()
bow = ManualBoW()
bow_matrix = bow.fit_transform(sms_sample)

print(f"Документов: {bow_matrix.shape[0]}, Признаков: {bow_matrix.shape[1]}")
print(f"\nДокумент 0: «{sms_sample[0]}»")
print(f"Вектор (ненулевые):")
for j in np.where(bow_matrix[0] > 0)[0]:
    print(f"  {bow.vocab_[j]:20s} = {bow_matrix[0, j]}")

Документов: 6, Признаков: 23

Документ 0: «hello after work tomorrow»
Вектор (ненулевые):
  after                = 1
  hello                = 1
  tomorrow             = 1
  work                 = 1


In [6]:
# --- Сравнение ручного BoW со sklearn ---
sklearn_bow = CountVectorizer(lowercase=True)
sklearn_bow_matrix = sklearn_bow.fit_transform(sms_sample).toarray()

assert bow.vocab_ == sklearn_bow.get_feature_names_out().tolist(), "Словари не совпали!"

max_diff = np.max(np.abs(bow_matrix - sklearn_bow_matrix))
print(f"Максимальное отклонение ручного BoW от sklearn: {max_diff}")

if max_diff == 0:
    print("Матрицы идентичны — ручная реализация совпадает со sklearn ✓")
else:
    print("ВНИМАНИЕ: есть расхождение — проверь формулы")

Максимальное отклонение ручного BoW от sklearn: 0
Матрицы идентичны — ручная реализация совпадает со sklearn ✓


## Шаг 2: Теперь то же через sklearn — и сравним accuracy

Мы убедились, что наш ручной BoW даёт ту же матрицу, что и `CountVectorizer`.
Теперь обучим классификатор и увидим, что accuracy тоже одинаковая.

Почему BoW здесь вообще работает? Для спама достаточно знать **какие слова** есть —
маркеры вроде `free`, `urgent`, `claim`, `bonus`. Порядок не важен.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_sms["text"],
    df_sms["label"],
    test_size=0.25,
    random_state=42,
    stratify=df_sms["label"],
)

# --- Ручная реализация ---
bow_full = ManualBoW()
manual_bow_train = bow_full.fit_transform(X_train.tolist())
manual_bow_test = bow_full.transform(X_test.tolist())

model_bow_manual = MultinomialNB()
model_bow_manual.fit(manual_bow_train, y_train)
pred_manual = model_bow_manual.predict(manual_bow_test)
acc_bow_manual = accuracy_score(y_test, pred_manual)

# --- sklearn CountVectorizer ---
vectorizer_bow = CountVectorizer(lowercase=True)
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)

model_bow = MultinomialNB()
model_bow.fit(X_train_bow, y_train)
pred_test = model_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, pred_test)

print("=" * 55)
print(f"Ручной BoW + MultinomialNB accuracy:  {acc_bow_manual:.4f}")
print(f"sklearn BoW + MultinomialNB accuracy:  {acc_bow:.4f}")
print("=" * 55)
print()
print(classification_report(y_test, pred_test))


In [ ]:
counts = X_train_bow.sum(axis=0).A1
feature_names = np.array(vectorizer_bow.get_feature_names_out())
ix = np.argsort(counts)[-15:][::-1]

print("Самые частые токены в обучении:")
for i in ix:
    print(feature_names[i], int(counts[i]))


In [ ]:
# ---------- Проверка корректности задачи 1 ----------

assert acc_bow > 0.80, (
    f"Accuracy слишком низкая ({acc_bow:.4f}). "
    "Проверь, что CountVectorizer обучен только на train, "
    "а predict вызван на тестовых данных."
)

assert set(model_bow.classes_) == {"spam", "ham"}, (
    f"Ожидали классы {{'spam', 'ham'}}, получили {set(model_bow.classes_)}. "
    "Проверь метки в CSV."
)

train_ratio = X_train_bow.shape[0] / (X_train_bow.shape[0] + X_test_bow.shape[0])
assert 0.6 < train_ratio < 0.9, (
    f"Доля обучающей выборки = {train_ratio:.2f}. Ожидается 0.70–0.80. "
    "Возможно, test_size задан некорректно в train_test_split."
)

print("Задача 1: все проверки пройдены ✓")

### Частые ошибки в задаче 1

**Ошибка 1: Использование TfidfVectorizer вместо CountVectorizer**

MultinomialNB ожидает неотрицательные счётчики (частоты слов).
TF-IDF даёт вещественные веса — формально MultinomialNB всё равно запустится,
но это нарушает его предположения. Для MultinomialNB используй `CountVectorizer`.

**Ошибка 2: Не выставлен `stratify` при split**
```python
# НЕПРАВИЛЬНО (классы могут распределиться неравномерно):
train_test_split(X, y, test_size=0.25)
# ПРАВИЛЬНО:
train_test_split(X, y, test_size=0.25, stratify=y)
```
На маленьких датасетах без стратификации один из классов может быть
сильно недопредставлен в тестовой выборке, и метрики будут нестабильными.

**Ошибка 3: Обучение и предсказание на одних и тех же данных**
```python
# НЕПРАВИЛЬНО:
model.fit(X_train_bow, y_train)
pred = model.predict(X_train_bow)   # оценка на train!
```
Accuracy на train всегда завышена. Оценивай только на `X_test`.

## Задача 2: Тональность отзывов

У нас есть короткие отзывы о товаре.
Для каждого отзыва нужно определить тональность:

- `positive`
- `negative`

Почему здесь естественен **TF-IDF**:

- в задаче важны отдельные слова-маркеры вроде `отлично`, `ужасно`, `брак`, `супер`;
- слишком частые служебные слова не должны сильно влиять на модель;
- логистическая регрессия на TF-IDF — это один из самых классических и сильных базлайнов для такой постановки.


In [ ]:
if not os.path.exists("reviews_small.csv"):
    rnd = random.Random(42)

    positive_templates = [
        "товар {adj} {noun}",
        "очень {adj} {noun}",
        "мне {verb} этот товар",
        "качество {adj} доставка {adj2}",
        "все {adj} буду брать еще",
    ]
    negative_templates = [
        "товар {adj} {noun}",
        "очень {adj} {noun}",
        "мне не {verb} этот товар",
        "качество {adj} доставка {adj2}",
        "все {adj} больше не куплю",
    ]

    pos_adj = ["отличный", "прекрасный", "качественный", "супер", "надежный"]
    neg_adj = ["ужасный", "плохой", "сломанный", "кошмарный", "дешевый"]
    pos_noun = ["вариант", "выбор", "товар", "заказ", "подарок"]
    neg_noun = ["брак", "мусор", "провал", "товар", "ужас"]
    pos_verb = ["понравился", "зашел", "подошел", "порадовал"]
    neg_verb = ["понравился", "зашел", "подошел", "порадовал"]
    pos_adj2 = ["быстрая", "аккуратная", "идеальная"]
    neg_adj2 = ["долгая", "ужасная", "плохая"]

    rows = []
    for _ in range(220):
        tpl = rnd.choice(positive_templates)
        text = tpl.format(
            adj=rnd.choice(pos_adj),
            noun=rnd.choice(pos_noun),
            verb=rnd.choice(pos_verb),
            adj2=rnd.choice(pos_adj2),
        )
        if rnd.random() < 0.25:
            text += " рекомендую"
        rows.append((text, "positive"))

    for _ in range(220):
        tpl = rnd.choice(negative_templates)
        text = tpl.format(
            adj=rnd.choice(neg_adj),
            noun=rnd.choice(neg_noun),
            verb=rnd.choice(neg_verb),
            adj2=rnd.choice(neg_adj2),
        )
        if rnd.random() < 0.25:
            text += " не рекомендую"
        rows.append((text, "negative"))

    rnd.shuffle(rows)
    pd.DataFrame(rows, columns=["text", "label"]).to_csv("reviews_small.csv", index=False)


In [ ]:
df_reviews = pd.read_csv("reviews_small.csv")
df_reviews.head()


## Шаг 1: Реализуем TF-IDF вручную

Прежде чем вызывать `TfidfVectorizer`, разберём, **что именно** он считает.

**TF (Term Frequency)** — сколько раз слово встретилось в документе:

$$\text{TF}(t, d) = \text{count}(t, d)$$

**IDF (Inverse Document Frequency)** — насколько слово редкое в корпусе:

$$\text{IDF}(t) = \ln\!\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

где $N$ — количество документов, $\text{df}(t)$ — в скольких документах встретилось слово $t$.

**TF-IDF** = TF × IDF, затем каждая строка нормируется по L2.

Реализуем это шаг за шагом на чистом numpy.

In [ ]:
class ManualTfidf:
    """TF-IDF с нуля — повторяет логику sklearn TfidfVectorizer(smooth_idf=True, norm='l2').

    Два метода:
      fit_transform(texts)  — построить словарь + IDF по texts, вернуть матрицу
      transform(texts)      — применить готовый словарь + IDF к новым текстам
    """

    def _tokenize(self, texts):
        return [text.lower().split() for text in texts]

    def fit_transform(self, texts):
        tokenized = self._tokenize(texts)

        # 1. Словарь: собираем уникальные слова, сортируем (как sklearn)
        self.vocab_ = sorted({w for tokens in tokenized for w in tokens})
        self.word_to_idx_ = {w: i for i, w in enumerate(self.vocab_)}
        n_docs = len(tokenized)

        # 2. TF — сколько раз слово встретилось в документе
        tf = self._count(tokenized)

        # 3. DF + IDF: запоминаем IDF, чтобы потом применить к тесту
        df = np.sum(tf > 0, axis=0)
        self.idf_ = np.log((1 + n_docs) / (1 + df)) + 1

        return self._apply_idf_and_norm(tf)

    def transform(self, texts):
        tokenized = self._tokenize(texts)
        tf = self._count(tokenized)
        return self._apply_idf_and_norm(tf)

    def _count(self, tokenized):
        tf = np.zeros((len(tokenized), len(self.vocab_)))
        for i, tokens in enumerate(tokenized):
            for w in tokens:
                if w in self.word_to_idx_:
                    tf[i, self.word_to_idx_[w]] += 1
        return tf

    def _apply_idf_and_norm(self, tf):
        tfidf = tf * self.idf_
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return tfidf / norms


# --- Демонстрация на маленькой выборке ---
texts_sample = df_reviews["text"].head(6).tolist()
m = ManualTfidf()
manual_matrix = m.fit_transform(texts_sample)

print(f"Документов: {manual_matrix.shape[0]}, Признаков (слов): {manual_matrix.shape[1]}")
print(f"\nСловарь (первые 15): {m.vocab_[:15]}")
print(f"\nTF-IDF вектор документа 0 (ненулевые):")
for j in np.where(manual_matrix[0] > 0)[0]:
    print(f"  {m.vocab_[j]:20s} = {manual_matrix[0, j]:.4f}")

In [ ]:
sklearn_vec = TfidfVectorizer(lowercase=True)
sklearn_matrix = sklearn_vec.fit_transform(texts_sample).toarray()
sklearn_vocab = sklearn_vec.get_feature_names_out().tolist()

assert m.vocab_ == sklearn_vocab, "Словари не совпали!"

max_diff = np.max(np.abs(manual_matrix - sklearn_matrix))
print(f"Максимальное отклонение ручного TF-IDF от sklearn: {max_diff:.2e}")

if max_diff < 1e-10:
    print("Матрицы идентичны — ручная реализация совпадает со sklearn ✓")
else:
    print("ВНИМАНИЕ: есть расхождение — проверь формулы")

print(f"\nСравнение для документа 0, слово '{m.vocab_[0]}':")
print(f"  Ручная:  {manual_matrix[0, 0]:.6f}")
print(f"  sklearn: {sklearn_matrix[0, 0]:.6f}")

## Шаг 2: Теперь то же самое через sklearn

Мы только что вычислили TF-IDF руками и убедились, что результат **совпадает** со sklearn.

Зачем тогда библиотека? `TfidfVectorizer`:
- работает на sparse-матрицах (экономит память);
- автоматически строит словарь на `fit`, а при `transform` использует готовый;
- позволяет задать `ngram_range`, `max_features`, стоп-слова — одной строкой.

Теперь обучим полный пайплайн: `TfidfVectorizer` → `LogisticRegression`,
и сравним accuracy с тем, что даёт наша ручная матрица.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_reviews["text"],
    df_reviews["label"],
    test_size=0.25,
    random_state=42,
    stratify=df_reviews["label"],
)

# --- Ручная реализация: fit на train, transform на test ---
m_full = ManualTfidf()
manual_train = m_full.fit_transform(X_train.tolist())
manual_test = m_full.transform(X_test.tolist())

model_manual = LogisticRegression(max_iter=2000, random_state=42)
model_manual.fit(manual_train, y_train)
pred_manual = model_manual.predict(manual_test)
acc_manual = accuracy_score(y_test, pred_manual)

# --- sklearn TfidfVectorizer ---
vectorizer_tfidf = TfidfVectorizer(lowercase=True)
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

model_tfidf = LogisticRegression(max_iter=2000, random_state=42)
model_tfidf.fit(X_train_tfidf, y_train)
pred_test = model_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, pred_test)

print("=" * 55)
print(f"Ручной TF-IDF + LogReg accuracy:  {acc_manual:.4f}")
print(f"sklearn TF-IDF + LogReg accuracy:  {acc_tfidf:.4f}")
print("=" * 55)
print()
print("Вывод: результаты совпадают, потому что формулы одинаковые.")
print("sklearn просто делает это быстрее и на sparse-матрицах.\n")
print(classification_report(y_test, pred_test))


In [ ]:
feature_names = np.array(vectorizer_tfidf.get_feature_names_out())
coefs = model_tfidf.coef_[0]

# В бинарной задаче знак коэффициента зависит от порядка классов.
# Посмотрим, какие слова сильнее двигают решение.
ix_top = np.argsort(coefs)[-10:][::-1]
ix_bottom = np.argsort(coefs)[:10]

print("Слова с самыми большими положительными коэффициентами:")
for i in ix_top:
    print(feature_names[i], round(coefs[i], 3))

print("\nСлова с самыми маленькими коэффициентами:")
for i in ix_bottom:
    print(feature_names[i], round(coefs[i], 3))


In [ ]:
# ---------- Проверка корректности задачи 2 ----------

assert acc_tfidf > 0.75, (
    f"Accuracy слишком низкая ({acc_tfidf:.4f}). "
    "Возможные причины: vectorizer обучен на тестовых данных, "
    "или перепутаны X и y при обучении."
)

pos_class = model_tfidf.classes_[1] if model_tfidf.classes_[1] == "positive" else model_tfidf.classes_[0]
assert set(model_tfidf.classes_) == {"positive", "negative"}, (
    f"Ожидали классы {{'positive', 'negative'}}, получили {set(model_tfidf.classes_)}. "
    "Проверь, что метки в CSV корректны."
)

assert X_train_tfidf.shape[0] != X_test_tfidf.shape[0], (
    "Размеры train и test совпадают — возможно, ты использовал одни и те же данные. "
    "Убедись, что train_test_split вызван корректно."
)

print("Задача 2: все проверки пройдены ✓")

### Частые ошибки в задаче 2

**Ошибка 1: `fit_transform` на тестовых данных**
```python
# НЕПРАВИЛЬНО:
X_test_tfidf = vectorizer_tfidf.fit_transform(X_test)
# ПРАВИЛЬНО:
X_test_tfidf = vectorizer_tfidf.transform(X_test)
```
Если вызвать `fit_transform` на тесте, словарь перестроится под тестовые данные,
и признаковое пространство train/test будет разным. Модель будет работать на случайном уровне.

**Ошибка 2: Обучение vectorizer на всех данных до split**
```python
# НЕПРАВИЛЬНО:
vectorizer = TfidfVectorizer()
X_all = vectorizer.fit_transform(all_texts)  # утечка данных!
X_train, X_test = ...
```
Это **data leakage**: IDF-веса вычислены с учётом тестовых документов,
и метрики на тесте будут завышены.

**Ошибка 3: Забыл `lowercase=True` или не почистил текст**

Слова `Отличный` и `отличный` будут разными признаками.
TfidfVectorizer по умолчанию приводит к нижнему регистру, но если ты делаешь
свою предобработку — убедись, что lowercase применён.

## Задача 3: Классификация намерения по короткой фразе

Есть короткие пользовательские запросы в поддержку магазина.
Нужно определить намерение:

- `order`
- `cancel`
- `delivery`
- `complaint`

Почему здесь важны **n-граммы поверх TF-IDF**:

- фразы очень короткие;
- важно не только наличие слов, но и их сочетания;
- биграммы вроде `отменить заказ`, `где доставка`, `не пришел`, `хочу вернуть`
  часто несут больше смысла, чем слова по отдельности.


In [ ]:
if not os.path.exists("intents_small.csv"):
    rnd = random.Random(44)

    templates = {
        "order": [
            "хочу заказать товар",
            "оформить заказ на товар",
            "как сделать заказ",
            "помогите заказать товар",
            "хочу оформить покупку",
        ],
        "cancel": [
            "хочу отменить заказ",
            "как отменить заказ",
            "отмена моего заказа",
            "нужно отменить покупку",
            "помогите отменить заказ",
        ],
        "delivery": [
            "где моя доставка",
            "когда привезут заказ",
            "статус доставки заказа",
            "почему задержка доставки",
            "когда будет курьер",
        ],
        "complaint": [
            "товар пришел сломанный",
            "хочу оставить жалобу",
            "мне прислали брак",
            "очень плохое качество",
            "товар не работает",
        ],
    }

    noise = ["срочно", "пожалуйста", "сегодня", "по заказу", "по товару", ""]

    rows = []
    for label, phrases in templates.items():
        for _ in range(180):
            text = rnd.choice(phrases)
            extra = rnd.choice(noise)
            text = (text + " " + extra).strip()
            rows.append((text, label))

    rnd.shuffle(rows)
    pd.DataFrame(rows, columns=["text", "label"]).to_csv("intents_small.csv", index=False)


In [ ]:
df_intents = pd.read_csv("intents_small.csv")
df_intents.head()


## Шаг 1: Что такое n-граммы и как их извлечь вручную

**Unigram** — отдельное слово: `хочу`, `отменить`, `заказ`.

**Bigram** — пара соседних слов: `хочу отменить`, `отменить заказ`.

Биграммы сохраняют **локальный порядок слов**. Это критично, когда
отдельные слова встречаются в разных интентах, а их сочетания — нет:

| Unigram | Где встречается |
|---------|----------------|
| `заказ` | order, cancel, delivery |
| `отменить` | cancel |
| `отменить заказ` (bigram) | **только cancel** |

Реализуем извлечение n-грамм вручную и построим TF-IDF поверх них.

In [ ]:
def extract_ngrams(tokens: List[str], n: int) -> List[str]:
    """Из списка токенов извлекает все n-граммы."""
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


class ManualNgramTfidf:
    """TF-IDF поверх n-грамм — от unigrams до заданного max_n."""

    def __init__(self, ngram_range=(1, 2)):
        self.ngram_range = ngram_range

    def _tokenize_and_ngram(self, texts):
        result = []
        for text in texts:
            tokens = text.lower().split()
            ngrams = []
            for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
                ngrams.extend(extract_ngrams(tokens, n))
            result.append(ngrams)
        return result

    def fit_transform(self, texts):
        tokenized = self._tokenize_and_ngram(texts)
        self.vocab_ = sorted({ng for doc in tokenized for ng in doc})
        self.ng_to_idx_ = {ng: i for i, ng in enumerate(self.vocab_)}
        n_docs = len(tokenized)

        tf = self._count(tokenized)
        df = np.sum(tf > 0, axis=0)
        self.idf_ = np.log((1 + n_docs) / (1 + df)) + 1
        return self._apply(tf)

    def transform(self, texts):
        tokenized = self._tokenize_and_ngram(texts)
        tf = self._count(tokenized)
        return self._apply(tf)

    def _count(self, tokenized):
        mat = np.zeros((len(tokenized), len(self.vocab_)))
        for i, ngrams in enumerate(tokenized):
            for ng in ngrams:
                if ng in self.ng_to_idx_:
                    mat[i, self.ng_to_idx_[ng]] += 1
        return mat

    def _apply(self, tf):
        tfidf = tf * self.idf_
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return tfidf / norms


# --- Демонстрация: посмотрим на n-граммы одной фразы ---
demo_text = "хочу отменить заказ пожалуйста"
demo_tokens = demo_text.lower().split()

print(f"Текст: «{demo_text}»\n")
print("Unigrams:", extract_ngrams(demo_tokens, 1))
print("Bigrams: ", extract_ngrams(demo_tokens, 2))
print("Trigrams:", extract_ngrams(demo_tokens, 3))

In [ ]:
# --- Сравнение ручных n-грамм со sklearn ---
intent_sample = df_intents["text"].head(8).tolist()

manual_ng = ManualNgramTfidf(ngram_range=(1, 2))
manual_ng_matrix = manual_ng.fit_transform(intent_sample)

sklearn_ng = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
sklearn_ng_matrix = sklearn_ng.fit_transform(intent_sample).toarray()
sklearn_ng_vocab = sklearn_ng.get_feature_names_out().tolist()

assert manual_ng.vocab_ == sklearn_ng_vocab, "Словари n-грамм не совпали!"

max_diff = np.max(np.abs(manual_ng_matrix - sklearn_ng_matrix))
print(f"Максимальное отклонение ручного N-gram TF-IDF от sklearn: {max_diff:.2e}")

if max_diff < 1e-10:
    print("Матрицы идентичны ✓")

print(f"\nРучных признаков (uni+bi): {len(manual_ng.vocab_)}")
print(f"sklearn признаков:         {len(sklearn_ng_vocab)}")

print(f"\nПримеры биграмм в словаре:")
bigrams = [ng for ng in manual_ng.vocab_ if " " in ng]
for bg in bigrams[:10]:
    print(f"  {bg}")

## Шаг 2: Обучим классификатор — ручная реализация vs sklearn, unigram vs bigram

Мы уже убедились, что наша ручная реализация n-грамм даёт те же числа, что sklearn.

Теперь интересный эксперимент: обучим **4 модели** и сравним, как
добавление биграмм влияет на accuracy.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_intents["text"],
    df_intents["label"],
    test_size=0.25,
    random_state=42,
    stratify=df_intents["label"],
)

# --- Ручная реализация: unigram vs bigram ---
m_uni = ManualNgramTfidf(ngram_range=(1, 1))
m_uni_train = m_uni.fit_transform(X_train.tolist())
m_uni_test = m_uni.transform(X_test.tolist())
model_m_uni = LogisticRegression(max_iter=2000, random_state=42)
model_m_uni.fit(m_uni_train, y_train)
acc_m_uni = accuracy_score(y_test, model_m_uni.predict(m_uni_test))

m_bi = ManualNgramTfidf(ngram_range=(1, 2))
m_bi_train = m_bi.fit_transform(X_train.tolist())
m_bi_test = m_bi.transform(X_test.tolist())
model_m_bi = LogisticRegression(max_iter=2000, random_state=42)
model_m_bi.fit(m_bi_train, y_train)
acc_m_bi = accuracy_score(y_test, model_m_bi.predict(m_bi_test))

# --- sklearn ---
vec_uni = TfidfVectorizer(ngram_range=(1, 1), lowercase=True)
X_train_uni = vec_uni.fit_transform(X_train)
X_test_uni = vec_uni.transform(X_test)
model_uni = LogisticRegression(max_iter=2000, random_state=42)
model_uni.fit(X_train_uni, y_train)
pred_uni = model_uni.predict(X_test_uni)
acc_uni = accuracy_score(y_test, pred_uni)

vec_bi = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X_train_bi = vec_bi.fit_transform(X_train)
X_test_bi = vec_bi.transform(X_test)
model_bi = LogisticRegression(max_iter=2000, random_state=42)
model_bi.fit(X_train_bi, y_train)
pred_bi = model_bi.predict(X_test_bi)
acc_bi = accuracy_score(y_test, pred_bi)

print("=" * 55)
print(f"{'Метод':<35s} {'Accuracy':>8s}")
print("-" * 55)
print(f"{'Ручной unigram TF-IDF':<35s} {acc_m_uni:>8.4f}")
print(f"{'sklearn unigram TF-IDF':<35s} {acc_uni:>8.4f}")
print(f"{'Ручной uni+bigram TF-IDF':<35s} {acc_m_bi:>8.4f}")
print(f"{'sklearn uni+bigram TF-IDF':<35s} {acc_bi:>8.4f}")
print("=" * 55)
print(f"\nДобавление биграмм: {acc_uni:.4f} → {acc_bi:.4f}")
print(f"Признаков: unigram={X_train_uni.shape[1]}, +bigram={X_train_bi.shape[1]}")


In [ ]:
print(classification_report(y_test, pred_bi))


In [ ]:
# ---------- Проверка корректности задачи 3 ----------

assert acc_uni > 0.60, (
    f"Unigram accuracy слишком низкая ({acc_uni:.4f}). "
    "Убедись, что TfidfVectorizer обучен на train и stratify задан в split."
)
assert acc_bi > 0.60, (
    f"Bigram accuracy слишком низкая ({acc_bi:.4f}). "
    "Проверь ngram_range=(1, 2) и что модель обучена на корректных данных."
)

n_classes = len(set(y_test))
assert n_classes >= 3, (
    f"Найдено только {n_classes} класс(а) в тесте. "
    "Проверь, что все интенты попали в обучение и test."
)

n_bigram_features = X_train_bi.shape[1]
n_unigram_features = X_train_uni.shape[1]
assert n_bigram_features > n_unigram_features, (
    f"Кол-во признаков с биграммами ({n_bigram_features}) не больше, чем без ({n_unigram_features}). "
    "Вероятно, ngram_range=(1, 2) не был задан."
)

print(f"Задача 3: все проверки пройдены ✓")
print(f"  Unigram features: {n_unigram_features}, Bigram features: {n_bigram_features}")
print(f"  Accuracy: unigram={acc_uni:.4f}, bigram={acc_bi:.4f}")

### Частые ошибки в задаче 3

**Ошибка 1: Использование слишком высоких n-грамм**
```python
# НЕПРАВИЛЬНО (для маленького корпуса):
TfidfVectorizer(ngram_range=(1, 5))
```
На маленьком датасете высокие n-граммы (3, 4, 5) создают огромное количество
уникальных, но редких признаков. Модель переобучается: accuracy на train ≈ 1.0,
на test резко падает. Для коротких текстов обычно достаточно (1, 2) или (1, 3).

**Ошибка 2: Сравнение моделей на разных split'ах**
```python
# НЕПРАВИЛЬНО:
X_train1, X_test1, ... = train_test_split(..., random_state=1)
X_train2, X_test2, ... = train_test_split(..., random_state=2)
```
Если split разный, ты сравниваешь не методы, а случайные разбиения.
Всегда фиксируй `random_state` одинаковым для обоих экспериментов.

**Ошибка 3: Не использовать `stratify` при мультиклассовой задаче**

Без стратификации некоторые классы могут полностью отсутствовать в тестовой выборке,
что приведёт к `UndefinedMetricWarning` в classification_report и нерепрезентативным метрикам.

## Задача 4: Классификация новостей через Word2Vec

Есть короткие новости по темам:

- `sport`
- `business`
- `politics`
- `tech`

Почему здесь естественен **Word2Vec**:

- нас интересуют не только частоты слов, но и **сходство контекстов**;
- слова из одной темы должны получить близкие векторы;
- затем можно получить вектор всего текста, усреднив эмбеддинги слов.

Это очень классическая постановка для Word2Vec:

1. обучить эмбеддинги слов;
2. превратить документ в средний embedding;
3. обучить обычный классификатор поверх эмбеддингов документов.


In [ ]:
if not os.path.exists("news_small.csv"):
    rnd = random.Random(45)

    topics = {
        "sport": {
            "subj": ["команда", "клуб", "тренер", "игрок", "нападающий"],
            "verb": ["выиграл", "проиграл", "обыграл", "подписал", "забил"],
            "obj": ["матч", "турнир", "финал", "чемпионат", "сезон"],
        },
        "business": {
            "subj": ["компания", "банк", "рынок", "инвестор", "фонд"],
            "verb": ["купил", "продал", "увеличил", "снизил", "объявил"],
            "obj": ["акции", "прибыль", "выручку", "долю", "контракт"],
        },
        "politics": {
            "subj": ["министр", "парламент", "президент", "депутат", "правительство"],
            "verb": ["обсудил", "подписал", "предложил", "утвердил", "объявил"],
            "obj": ["закон", "реформу", "санкции", "бюджет", "решение"],
        },
        "tech": {
            "subj": ["стартап", "разработчик", "компания", "инженер", "платформа"],
            "verb": ["запустил", "обновил", "показал", "создал", "выпустил"],
            "obj": ["модель", "сервис", "приложение", "чип", "прототип"],
        },
    }

    rows = []
    for label, parts in topics.items():
        for _ in range(220):
            text = f"{rnd.choice(parts['subj'])} {rnd.choice(parts['verb'])} {rnd.choice(parts['obj'])}"
            if rnd.random() < 0.4:
                text += " сегодня"
            if rnd.random() < 0.25:
                text += " официально"
            rows.append((text, label))

    rnd.shuffle(rows)
    pd.DataFrame(rows, columns=["text", "label"]).to_csv("news_small.csv", index=False)


In [ ]:
df_news = pd.read_csv("news_small.csv")
df_news.head()


## Подготовка токенов

Word2Vec учится на последовательностях токенов.
Поэтому сначала приведём тексты к простому токенизированному виду.


In [ ]:
# При необходимости установи gensim:
# !pip install gensim


In [ ]:
from gensim.models import Word2Vec

def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-zа-яё0-9 ]", " ", text)
    return [t for t in text.split() if t]

news_tokens = df_news["text"].map(simple_tokenize).tolist()


## Шаг 1: Идея Word2Vec — контекстные окна и обучение эмбеддингов

Word2Vec учит **плотные векторы** слов, исходя из принципа:
> *Слова, встречающиеся в похожих контекстах, должны иметь похожие векторы.*

### Skip-gram: из центрального слова предсказываем контекстные

Для каждого слова в тексте берём **окно** размером `window`:

```
текст:   [команда] [выиграла] [матч] [сегодня]
                       ↑ центральное слово (window=1)
          ← контекст →   ← контекст →
```

Из этого получаются пары `(центральное, контекстное)`:
- `(выиграла, команда)`
- `(выиграла, матч)`

Модель учится: для центрального слова предсказать контекстное через
два слоя (embedding → projection), минимизируя cross-entropy.

Ниже — упрощённая реализация на numpy, которая показывает механику.

In [ ]:
def build_skipgram_pairs(tokenized_docs, window=2):
    """Из корпуса документов строит пары (центральное_слово, контекстное_слово)."""
    pairs = []
    for tokens in tokenized_docs:
        for i, center in enumerate(tokens):
            left = max(0, i - window)
            right = min(len(tokens), i + window + 1)
            for j in range(left, right):
                if j != i:
                    pairs.append((center, tokens[j]))
    return pairs


# Демонстрация на одном предложении
demo_tokens = ["команда", "выиграла", "матч", "сегодня"]
demo_pairs = build_skipgram_pairs([demo_tokens], window=2)

print(f"Текст: {demo_tokens}\n")
print(f"Skip-gram пары (window=2):")
for center, context in demo_pairs:
    print(f"  ({center}, {context})")
print(f"\nВсего пар: {len(demo_pairs)}")

In [ ]:
class SimpleSkipGram:
    """Упрощённый Skip-Gram на numpy — показывает механику обучения Word2Vec.

    Два слоя:
      W_in  (vocab_size × embed_dim)  — embedding центрального слова
      W_out (embed_dim × vocab_size)  — projection на контекстные слова

    Для каждой пары (center, context): softmax по словарю, cross-entropy loss.
    На больших словарях используют negative sampling, но здесь словарь маленький.
    """

    def __init__(self, vocab_size, embed_dim=10, lr=0.05, seed=42):
        rng = np.random.RandomState(seed)
        self.W_in = rng.randn(vocab_size, embed_dim).astype(np.float32) * 0.1
        self.W_out = rng.randn(embed_dim, vocab_size).astype(np.float32) * 0.1
        self.lr = lr
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def _softmax(self, logits):
        e = np.exp(logits - logits.max())
        return e / e.sum()

    def train_pair(self, center_idx, context_idx):
        h = self.W_in[center_idx]               # (embed_dim,)
        logits = h @ self.W_out                  # (vocab_size,)
        probs = self._softmax(logits)            # (vocab_size,)

        loss = -np.log(probs[context_idx] + 1e-12)

        # Градиент по W_out и h
        grad_out = probs.copy()
        grad_out[context_idx] -= 1               # (vocab_size,)
        dW_out = np.outer(h, grad_out)            # (embed_dim, vocab_size)
        dh = self.W_out @ grad_out                # (embed_dim,)

        self.W_out -= self.lr * dW_out
        self.W_in[center_idx] -= self.lr * dh
        return loss

    def train_epoch(self, pairs_idx):
        total_loss = 0
        np.random.shuffle(pairs_idx)
        for c, ctx in pairs_idx:
            total_loss += self.train_pair(c, ctx)
        return total_loss / len(pairs_idx)

    @property
    def embeddings(self):
        return self.W_in


# Строим словарь по нашим новостям
all_words = sorted({w for tokens in news_tokens for w in tokens})
w2i = {w: i for i, w in enumerate(all_words)}

# Строим skip-gram пары
sg_pairs = build_skipgram_pairs(news_tokens, window=2)
pairs_idx = np.array([(w2i[c], w2i[ctx]) for c, ctx in sg_pairs if c in w2i and ctx in w2i])

print(f"Словарь: {len(all_words)} слов")
print(f"Skip-gram пар: {len(pairs_idx)}")

# Обучаем упрощённый Skip-Gram
sg = SimpleSkipGram(vocab_size=len(all_words), embed_dim=20, lr=0.05)
for epoch in range(15):
    loss = sg.train_epoch(pairs_idx)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:2d}, loss: {loss:.4f}")

print(f"\nРазмер embedding-матрицы: {sg.embeddings.shape}")

In [ ]:
def most_similar_manual(word, top_n=5):
    if word not in w2i:
        return []
    vec = sg.embeddings[w2i[word]].reshape(1, -1)
    sims = cosine_similarity(vec, sg.embeddings)[0]
    idxs = np.argsort(sims)[::-1][1:top_n + 1]
    return [(all_words[i], round(sims[i], 3)) for i in idxs]

print("Ручной Skip-Gram — похожие слова:")
for test_word in ["команда", "компания", "министр", "стартап"]:
    if test_word in w2i:
        print(f"  {test_word}: {most_similar_manual(test_word)}")

## Шаг 2: Теперь то же через gensim Word2Vec — и сравним

Наша ручная реализация показала **механику**: skip-gram пары, softmax, градиенты.
Но gensim Word2Vec:
- использует **negative sampling** вместо полного softmax (быстрее на больших словарях);
- оптимизирован на C (в 100–1000× быстрее);
- имеет удобный API `.most_similar()`.

Обучим gensim и сравним, какие слова он считает похожими.

In [ ]:
w2v = Word2Vec(
    sentences=news_tokens,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    epochs=50,
    seed=42,
)

print("gensim Word2Vec — похожие слова:")
for test_word in ["команда", "компания", "министр", "стартап"]:
    if test_word in w2v.wv:
        sims = w2v.wv.most_similar(test_word, topn=5)
        print(f"  {test_word}: {[(w, round(s, 3)) for w, s in sims]}")

print("\nРучной Skip-Gram находит похожие тематические слова,")
print("но gensim даёт более стабильные результаты благодаря negative sampling и большему числу эпох.")


## Вектор документа как среднее эмбеддингов слов

Это простой, но очень классический baseline:

- каждому слову соответствует embedding;
- документ представляется как среднее embedding'ов его слов;
- дальше на этих признаках обучается обычная логистическая регрессия.


In [ ]:
def mean_embedding(tokens: List[str], model: Word2Vec) -> np.ndarray:
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

def mean_embedding_manual(tokens: List[str], emb_matrix, word2idx, embed_dim):
    vecs = [emb_matrix[word2idx[t]] for t in tokens if t in word2idx]
    if not vecs:
        return np.zeros(embed_dim, dtype=np.float32)
    return np.mean(vecs, axis=0)

y_news = df_news["label"].to_numpy()

# --- Ручные эмбеддинги ---
X_news_manual = np.vstack([
    mean_embedding_manual(toks, sg.embeddings, w2i, sg.embed_dim) for toks in news_tokens
])
X_tr_m, X_te_m, y_tr, y_te = train_test_split(
    X_news_manual, y_news, test_size=0.25, random_state=42, stratify=y_news
)
model_manual_w2v = LogisticRegression(max_iter=2000, random_state=42)
model_manual_w2v.fit(X_tr_m, y_tr)
acc_manual_w2v = accuracy_score(y_te, model_manual_w2v.predict(X_te_m))

# --- gensim эмбеддинги ---
X_news = np.vstack([mean_embedding(toks, w2v) for toks in news_tokens])
X_train, X_test, y_train, y_test = train_test_split(
    X_news, y_news, test_size=0.25, random_state=42, stratify=y_news
)
model_news = LogisticRegression(max_iter=2000, random_state=42)
model_news.fit(X_train, y_train)
pred_news = model_news.predict(X_test)
acc_gensim_w2v = accuracy_score(y_test, pred_news)

print("=" * 60)
print(f"Ручной Skip-Gram (dim=20, 15 эпох) + LogReg:  {acc_manual_w2v:.4f}")
print(f"gensim Word2Vec  (dim=50, 50 эпох) + LogReg:  {acc_gensim_w2v:.4f}")
print("=" * 60)
print()
print("gensim лучше, потому что: больше размерность, больше эпох,")
print("negative sampling вместо полного softmax.\n")
print(classification_report(y_test, pred_news))


In [ ]:
# ---------- Проверка корректности задачи 4 ----------

acc_w2v = accuracy_score(y_test, pred_news)

assert acc_w2v > 0.50, (
    f"Accuracy Word2Vec модели слишком низкая ({acc_w2v:.4f}). "
    "Убедись, что эмбеддинги обучены на токенах train, "
    "и что mean_embedding корректно обрабатывает OOV-слова."
)

assert X_train.shape[1] == w2v.vector_size, (
    f"Размерность признаков ({X_train.shape[1]}) не совпадает с vector_size ({w2v.vector_size}). "
    "Проверь, что mean_embedding возвращает вектор нужной размерности."
)

zero_rows_train = np.all(X_train == 0, axis=1).sum()
if zero_rows_train > 0:
    print(f"  ВНИМАНИЕ: {zero_rows_train} документ(ов) в train имеют нулевой вектор (все слова OOV).")

n_topics = len(set(y_test))
assert n_topics >= 3, (
    f"В тесте только {n_topics} тем(ы). Проверь, что stratify задан в split."
)

print(f"Задача 4: все проверки пройдены ✓")
print(f"  Word2Vec vocab size: {len(w2v.wv)}")
print(f"  Accuracy: {acc_w2v:.4f}")

### Частые ошибки в задаче 4

**Ошибка 1: Не обрабатывать OOV (out-of-vocabulary) слова**
```python
# НЕПРАВИЛЬНО (KeyError на незнакомом слове):
def mean_embedding(tokens, model):
    vecs = [model.wv[t] for t in tokens]
    return np.mean(vecs, axis=0)

# ПРАВИЛЬНО:
def mean_embedding(tokens, model):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)
```
В тестовых данных могут встречаться слова, которых не было в train.
Без проверки `if t in model.wv` код упадёт с ошибкой.

**Ошибка 2: Использование суммы вместо среднего**
```python
# НЕОСТОРОЖНО:
return np.sum(vecs, axis=0)
```
Сумма зависит от длины документа: длинные тексты получат непропорционально
большие векторы. Среднее нормализует по длине и даёт стабильные признаки.

**Ошибка 3: Слишком маленькое `epochs` или `vector_size`**

На маленьких корпусах Word2Vec нужно больше эпох (30–50), чтобы веса сошлись.
А `vector_size=10` может быть слишком мал для 4 тем — 50 обычно хороший выбор.

**Ошибка 4: Обучение Word2Vec на всех данных (train + test)**
```python
# НЕПРАВИЛЬНО (data leakage):
all_tokens = [tokenize(t) for t in all_texts]
w2v = Word2Vec(sentences=all_tokens, ...)
```
Эмбеддинги должны обучаться только на train. Иначе модель видит контексты тестовых слов.

## Задача 5: Определение языка короткого текста через fastText

Есть короткие фразы на разных языках.
Нужно определить язык:

- `ru`
- `en`
- `es`
- `de`

Почему это классическая задача для **fastText**:

- fastText учитывает **символьные подслова**;
- это особенно полезно на коротких текстах;
- модель хорошо переносит редкие слова, морфологию и небольшие опечатки;
- задача language identification — один из самых известных классических сценариев fastText.


In [ ]:
if not os.path.exists("lang_small.csv"):
    rnd = random.Random(46)

    phrases = {
        "ru": [
            "где мой заказ",
            "когда будет доставка",
            "мне нужна помощь",
            "товар не работает",
            "спасибо за ответ",
        ],
        "en": [
            "where is my order",
            "when will delivery arrive",
            "i need help",
            "the item is broken",
            "thanks for your answer",
        ],
        "es": [
            "donde esta mi pedido",
            "cuando llega la entrega",
            "necesito ayuda",
            "el producto no funciona",
            "gracias por tu respuesta",
        ],
        "de": [
            "wo ist meine bestellung",
            "wann kommt die lieferung",
            "ich brauche hilfe",
            "das produkt funktioniert nicht",
            "danke fur deine antwort",
        ],
    }

    rows = []
    for label, texts in phrases.items():
        for _ in range(180):
            text = rnd.choice(texts)
            if rnd.random() < 0.25:
                text = text.replace("delivery", "delivry").replace("заказ", "закас").replace("pedido", "peddo")
            rows.append((text, label))

    rnd.shuffle(rows)
    pd.DataFrame(rows, columns=["text", "label"]).to_csv("lang_small.csv", index=False)


In [ ]:
df_lang = pd.read_csv("lang_small.csv")
df_lang.head()


## Подготовка данных в формате fastText

Для supervised fastText каждая строка обычно имеет вид:

`__label__<класс> текст`

Дальше модель обучается напрямую предсказывать метку.


In [ ]:
# При необходимости установи fasttext:
# !pip install fasttext


In [ ]:
train_lang, test_lang = train_test_split(
    df_lang,
    test_size=0.25,
    random_state=42,
    stratify=df_lang["label"],
)

def to_fasttext_file(df: pd.DataFrame, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            line = f"__label__{row['label']} {row['text']}\n"
            f.write(line)

to_fasttext_file(train_lang, "lang_train_fasttext.txt")
to_fasttext_file(test_lang, "lang_test_fasttext.txt")


In [ ]:
import fasttext

ft_model = fasttext.train_supervised(
    input="lang_train_fasttext.txt",
    epoch=50,
    lr=0.8,
    wordNgrams=2,
    minn=2,
    maxn=5,
    dim=50,
)

print(ft_model.test("lang_test_fasttext.txt"))


In [ ]:
def predict_fasttext(texts: List[str], model) -> List[str]:
    preds = []
    for text in texts:
        labels, _ = model.predict(text)
        preds.append(labels[0].replace("__label__", ""))
    return preds

pred_lang = predict_fasttext(test_lang["text"].tolist(), ft_model)

print("fastText accuracy:", round(accuracy_score(test_lang["label"], pred_lang), 4))
print(classification_report(test_lang["label"], pred_lang))


In [ ]:
# ---------- Проверка корректности задачи 5 ----------

acc_ft = accuracy_score(test_lang["label"], pred_lang)

assert acc_ft > 0.70, (
    f"Accuracy fastText слишком низкая ({acc_ft:.4f}). "
    "Проверь формат файлов __label__<class> text и параметры обучения."
)

n_langs = len(set(test_lang["label"]))
assert n_langs >= 3, (
    f"В тесте только {n_langs} язык(а). Проверь данные и stratify."
)

assert len(pred_lang) == len(test_lang), (
    f"Кол-во предсказаний ({len(pred_lang)}) не совпадает с кол-вом тестовых примеров ({len(test_lang)}). "
    "Убедись, что предсказания сделаны для всех тестовых текстов."
)

pred_labels_set = set(pred_lang)
true_labels_set = set(test_lang["label"])
unexpected = pred_labels_set - true_labels_set
assert not unexpected, (
    f"Модель предсказала неожиданные метки: {unexpected}. "
    "Возможно, __label__ не был корректно удалён из предсказаний."
)

print(f"Задача 5: все проверки пройдены ✓")
print(f"  Accuracy: {acc_ft:.4f}, Языков: {n_langs}")

### Частые ошибки в задаче 5

**Ошибка 1: Неправильный формат файла для fastText**
```python
# НЕПРАВИЛЬНО (нет метки в начале строки):
f.write(f"{text} {label}\n")
# ПРАВИЛЬНО:
f.write(f"__label__{label} {text}\n")
```
fastText ожидает формат `__label__<class> <text>`. Если метка не в начале
или нет префикса `__label__`, модель не распарсит данные.

**Ошибка 2: Не удалить `__label__` из предсказаний**
```python
# НЕПРАВИЛЬНО:
labels, _ = model.predict(text)
pred = labels[0]  # "__label__ru" вместо "ru"
# ПРАВИЛЬНО:
pred = labels[0].replace("__label__", "")
```
fastText возвращает метки с префиксом. Если забыть его убрать,
`accuracy_score` сравнит `"__label__ru"` с `"ru"` и accuracy будет 0.

**Ошибка 3: Слишком мало эпох или слишком низкий learning rate**

Для коротких текстов и маленького датасета fastText нужно больше эпох
(25–50) и достаточно высокий lr (0.5–1.0). С дефолтными параметрами
(epoch=5, lr=0.1) модель может не сойтись.

**Ошибка 4: Не указан `wordNgrams` или `minn`/`maxn`**

Вся сила fastText — в subword information. Без `minn`/`maxn` модель
не использует символьные n-граммы и теряет своё главное преимущество
перед обычным BoW.

## Вывод

Для каждого метода мы прошли путь **от формулы до работающей модели**:

| Метод | Ручная реализация | Библиотека | Результат |
|-------|-------------------|------------|-----------|
| **Bag of Words** | Словарь + подсчёт | `CountVectorizer` | Матрицы совпадают |
| **TF-IDF** | TF × IDF + L2-норма | `TfidfVectorizer` | Матрицы совпадают |
| **N-граммы** | Извлечение пар + TF-IDF | `TfidfVectorizer(ngram_range)` | Матрицы совпадают |
| **Word2Vec** | Skip-gram + softmax + SGD | `gensim.Word2Vec` | Похожие слова ≈ совпадают |

Ключевые уроки:

1. **BoW** и **TF-IDF** — это простые линейные преобразования текста в вектор;
2. **n-граммы** добавляют локальный контекст, но увеличивают размерность;
3. **Word2Vec** учит плотные эмбеддинги из контекстных окон;
4. Библиотеки делают **то же самое**, но быстрее и в sparse-формате;
5. Понимание формул помогает отлавливать ошибки (data leakage, OOV, неправильный split).
